# ConvLSTM 弹跳小球时空预测实验

本实验以自生成的弹跳小球序列为对象，使用 ConvLSTM 根据前若干帧预测下一帧，并通过控制变量比较不同结构设计对预测误差的影响。


## 1. 实验设置

为保证比较可复现，所有实验固定随机种子 42。正式设置使用 2,000 条训练序列、200 条独立测试序列、5 个 epoch、批大小 64 和 Adam（学习率 0.001）。`QUICK_RUN=True` 仅用于运行前检查，不用于记录正式结论。


In [ ]:
from pathlib import Path
import csv
import json
import random
import time

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
RESULT_DIR = ROOT / 'results'
RESULT_DIR.mkdir(parents=True, exist_ok=True)

SEED = 42
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
QUICK_RUN = False  # 仅用于先检查流程；正式实验使用 False
N_TRAIN, N_TEST, EPOCHS = (800, 100, 3) if QUICK_RUN else (2000, 200, 5)
BATCH_SIZE = 64
LR = 1e-3
IMAGE_SIZE, SEQ_LEN = 32, 10

plt.rcParams['font.sans-serif'] = ['Microsoft YaHei', 'SimHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False
print(f'设备: {DEVICE}; 训练/测试: {N_TRAIN}/{N_TEST}; epoch: {EPOCHS}')


## 2. 数据生成与划分

每条序列包含 10 帧 32×32 二值图像。小球半径为 2，初速度在 0.8 到 1.6 像素/帧之间随机采样，撞到边界后反弹。训练集与测试集使用不同随机种子独立生成，避免样本重叠。


In [ ]:
def set_seed(seed: int = SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def make_sequences(n_seq, T=SEQ_LEN, size=IMAGE_SIZE, r=2, seed=SEED):
    """生成 (N,T,H,W,1) 的单球边界反弹序列。"""
    rng = np.random.default_rng(seed)
    yy, xx = np.mgrid[0:size, 0:size]
    seqs = np.zeros((n_seq, T, size, size), dtype=np.float32)
    for s in range(n_seq):
        x, y = rng.uniform(4, size - 5, 2)
        vx, vy = rng.choice([-1, 1], 2) * rng.uniform(0.8, 1.6, 2)
        for t in range(T):
            x, y = x + vx, y + vy
            if x < r or x > size - r:
                vx = -vx
                x = np.clip(x, r, size - r)
            if y < r or y > size - r:
                vy = -vy
                y = np.clip(y, r, size - r)
            seqs[s, t] = ((xx - x) ** 2 + (yy - y) ** 2 <= r ** 2)
    return seqs[..., None]


# 独立生成训练集与测试集，避免训练/测试样本重叠。
set_seed(SEED)
train_data = make_sequences(N_TRAIN, seed=SEED)
test_data = make_sequences(N_TEST, seed=SEED + 1)
print(train_data.shape, test_data.shape)


## 3. 基线 ConvLSTM

基线模型使用 1 层 ConvLSTM、32 个隐藏通道和 3×3 卷积核，读取前 4 帧预测第 5 帧。每个时间步将当前输入与隐藏状态沿通道维拼接，经一次卷积生成输入门、遗忘门、候选状态和输出门。


In [ ]:
class ConvLSTMCell(nn.Module):
    """合并四门的 ConvLSTM 单元。"""
    def __init__(self, in_ch, hid_ch, k=3):
        super().__init__()
        self.hid = hid_ch
        self.conv = nn.Conv2d(in_ch + hid_ch, 4 * hid_ch, k, padding=k // 2)

    def forward(self, x, state):
        h, c = state
        z = self.conv(torch.cat([x, h], dim=1))
        i, f, g, o = z.chunk(4, dim=1)
        i, f, o = torch.sigmoid(i), torch.sigmoid(f), torch.sigmoid(o)
        c_new = f * c + i * torch.tanh(g)
        h_new = o * torch.tanh(c_new)
        return h_new, c_new


class ConvLSTM(nn.Module):
    def __init__(self, in_ch=1, hid=32, k=3, layers=1):
        super().__init__()
        channels = [in_ch] + [hid] * layers
        self.cells = nn.ModuleList([ConvLSTMCell(channels[i], channels[i + 1], k)
                                    for i in range(layers)])
        self.out = nn.Conv2d(hid, 1, 3, padding=1)

    def forward(self, x):  # (B,T,C,H,W)
        b, _, _, h, w = x.shape
        states = [(torch.zeros(b, cell.hid, h, w, device=x.device, dtype=x.dtype),
                   torch.zeros(b, cell.hid, h, w, device=x.device, dtype=x.dtype))
                  for cell in self.cells]
        for t in range(x.shape[1]):
            xt = x[:, t]
            for j, cell in enumerate(self.cells):
                states[j] = cell(xt, states[j])
                xt = states[j][0]
        return self.out(states[-1][0]).squeeze(1)


class FlattenLSTM(nn.Module):
    """实验 5：每一帧展平为 1024 维的全连接 LSTM 对照。"""
    def __init__(self, image_size=IMAGE_SIZE, hidden_size=256):
        super().__init__()
        self.image_size = image_size
        features = image_size * image_size
        self.lstm = nn.LSTM(features, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, features)

    def forward(self, x):
        b, t, _, h, w = x.shape
        sequence = x.reshape(b, t, h * w)
        output, _ = self.lstm(sequence)
        return self.fc(output[:, -1]).reshape(b, h, w)


## 4. 训练、评估与可视化

训练过程中记录训练损失、测试 MSE 和测试 MAE。每组实验保存曲线、预测示例和指标文件，以便复核结果。


In [ ]:
def get_xy(data, input_frames):
    if input_frames >= data.shape[1]:
        raise ValueError('输入帧数必须小于序列总长度')
    x = torch.from_numpy(data[:, :input_frames]).permute(0, 1, 4, 2, 3).float()
    y = torch.from_numpy(data[:, input_frames, :, :, 0]).float()
    return x, y


def evaluate(model, loader):
    model.eval()
    sq_error = abs_error = count = 0.0
    predictions, targets = [], []
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            pred = model(x)
            sq_error += (pred - y).pow(2).sum().item()
            abs_error += (pred - y).abs().sum().item()
            count += y.numel()
            predictions.append(pred.cpu())
            targets.append(y.cpu())
    return sq_error / count, abs_error / count, torch.cat(predictions), torch.cat(targets)


def save_curve(history, path, title):
    fig, ax = plt.subplots(figsize=(7, 4))
    epochs = range(1, len(history['train_loss']) + 1)
    ax.plot(epochs, history['train_loss'], marker='o', label='训练损失')
    ax.plot(epochs, history['test_mse'], marker='s', label='测试 MSE')
    ax.set(xlabel='Epoch', ylabel='Loss / MSE', title=title)
    ax.grid(alpha=0.3); ax.legend(); fig.tight_layout()
    fig.savefig(path, dpi=180)
    plt.close(fig)


def save_prediction_figure(test_x, targets, predictions, path, input_frames, indices=(0, 1, 2)):
    cols = input_frames + 2
    fig, axes = plt.subplots(len(indices), cols, figsize=(1.7 * cols, 1.8 * len(indices)))
    axes = np.atleast_2d(axes)
    for row, idx in enumerate(indices):
        for t in range(input_frames):
            axes[row, t].imshow(test_x[idx, t, 0], cmap='gray', vmin=0, vmax=1)
            axes[row, t].set_title(f'输入 {t + 1}')
        mse = (predictions[idx] - targets[idx]).pow(2).mean().item()
        axes[row, -2].imshow(targets[idx], cmap='gray', vmin=0, vmax=1)
        axes[row, -2].set_title('真实下一帧')
        axes[row, -1].imshow(predictions[idx], cmap='gray', vmin=0, vmax=1)
        axes[row, -1].set_title(f'预测帧\nMSE={mse:.5f}')
        for ax in axes[row]: ax.axis('off')
    fig.tight_layout(); fig.savefig(path, dpi=180, bbox_inches='tight'); plt.close(fig)


In [ ]:
def run_experiment(config, run_seed=SEED, save_artifacts=True):
    """训练一组实验，返回可写入报告的真实指标。"""
    set_seed(run_seed)
    name = config['name']
    exp_dir = RESULT_DIR / config['id'] / f'seed_{run_seed}'
    exp_dir.mkdir(parents=True, exist_ok=True)
    input_frames = config.get('input_frames', 4)
    train_x, train_y = get_xy(train_data, input_frames)
    test_x, test_y = get_xy(test_data, input_frames)
    train_loader = DataLoader(TensorDataset(train_x, train_y), batch_size=BATCH_SIZE, shuffle=True)
    test_loader = DataLoader(TensorDataset(test_x, test_y), batch_size=BATCH_SIZE, shuffle=False)

    if config.get('model_type', 'convlstm') == 'flatten_lstm':
        model = FlattenLSTM()
    else:
        model = ConvLSTM(hid=config.get('hidden', 32), k=config.get('kernel', 3),
                         layers=config.get('layers', 1))
    model = model.to(DEVICE)
    loss_fn = nn.L1Loss() if config.get('loss', 'mse') == 'l1' else nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=LR)
    history = {'train_loss': [], 'test_mse': [], 'test_mae': []}
    started = time.perf_counter()
    for epoch in range(EPOCHS):
        model.train(); total_loss = 0.0
        for x, y in train_loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            optimizer.zero_grad()
            loss = loss_fn(model(x), y)
            loss.backward(); optimizer.step()
            total_loss += loss.item() * x.size(0)
        mse, mae, _, _ = evaluate(model, test_loader)
        history['train_loss'].append(total_loss / len(train_loader.dataset))
        history['test_mse'].append(mse); history['test_mae'].append(mae)
        print(f"{name} | epoch {epoch+1}/{EPOCHS}: train={history['train_loss'][-1]:.6f}, MSE={mse:.6f}, MAE={mae:.6f}")
    elapsed = time.perf_counter() - started
    mse, mae, preds, targets = evaluate(model, test_loader)
    result = {
        'id': config['id'], 'name': name, 'seed': run_seed, 'config': config,
        'test_mse': mse, 'test_mae': mae, 'seconds': elapsed,
        'parameters': sum(p.numel() for p in model.parameters()), 'history': history
    }
    if save_artifacts:
        save_curve(history, exp_dir / 'loss_curve.png', name)
        save_prediction_figure(test_x, targets, preds, exp_dir / 'prediction_examples.png', input_frames)
        with open(exp_dir / 'metrics.json', 'w', encoding='utf-8') as f:
            json.dump(result, f, ensure_ascii=False, indent=2)
        torch.save({'state_dict': model.cpu().state_dict(), 'config': config, 'result': result}, exp_dir / 'model.pt')
    return result, test_x, targets, preds


## 5. 基线与控制变量实验

以下配置依次运行基线以及六组对照实验。每组只修改一个变量：层数、隐藏通道、卷积核、输入帧数、网络结构或损失函数；其余训练条件保持不变。代码会输出 MSE、MAE、耗时和参数量。

基线单元的手算参数量为：ConvLSTM 四个门合计 38,144，输出卷积为 289，总参数量为 38,433。


In [ ]:
EXPERIMENTS = [
    {'id': 'baseline', 'name': '0 基线 Baseline', 'layers': 1, 'hidden': 32, 'kernel': 3, 'input_frames': 4, 'loss': 'mse'},
    {'id': 'exp1', 'name': '1 加深层数', 'layers': 2, 'hidden': 32, 'kernel': 3, 'input_frames': 4, 'loss': 'mse'},
    {'id': 'exp2', 'name': '2 隐藏通道 32→64', 'layers': 1, 'hidden': 64, 'kernel': 3, 'input_frames': 4, 'loss': 'mse'},
    {'id': 'exp3', 'name': '3 卷积核 3→5', 'layers': 1, 'hidden': 32, 'kernel': 5, 'input_frames': 4, 'loss': 'mse'},
    {'id': 'exp4', 'name': '4 输入帧数 4→8', 'layers': 1, 'hidden': 32, 'kernel': 3, 'input_frames': 8, 'loss': 'mse'},
    {'id': 'exp5', 'name': '5 结构对照 FlattenLSTM', 'model_type': 'flatten_lstm', 'input_frames': 4, 'loss': 'mse'},
    {'id': 'exp6', 'name': '6 损失函数 MSE→L1', 'layers': 1, 'hidden': 32, 'kernel': 3, 'input_frames': 4, 'loss': 'l1'},
]

BASELINE_CELL_PARAMS = 4 * (3 * 3 * 1 * 32 + 3 * 3 * 32 * 32 + 32)
BASELINE_OUT_PARAMS = 3 * 3 * 32 * 1 + 1
print(f'手算参数量：细胞={BASELINE_CELL_PARAMS}, 输出卷积={BASELINE_OUT_PARAMS}, 总计={BASELINE_CELL_PARAMS + BASELINE_OUT_PARAMS}')

all_results, cached_outputs = [], {}
for config in EXPERIMENTS:
    result, x, y, pred = run_experiment(config)
    all_results.append(result); cached_outputs[config['id']] = (x, y, pred)

# 报告规定的固定图片名；各组的独立副本仍保留在 results/<实验名>/seed_42/。
for config, result in zip(EXPERIMENTS, all_results):
    source = RESULT_DIR / config['id'] / f'seed_{SEED}' / 'loss_curve.png'
    target = RESULT_DIR / ('result_baseline.png' if config['id'] == 'baseline' else f"result_{config['id']}.png")
    target.write_bytes(source.read_bytes())

with open(RESULT_DIR / 'experiment_results.csv', 'w', newline='', encoding='utf-8-sig') as f:
    writer = csv.DictWriter(f, fieldnames=['id', 'name', 'seed', 'test_mse', 'test_mae', 'seconds', 'parameters', 'config'])
    writer.writeheader()
    for r in all_results:
        writer.writerow({**{k: r[k] for k in writer.fieldnames if k != 'config'}, 'config': json.dumps(r['config'], ensure_ascii=False)})
print('已保存:', RESULT_DIR / 'experiment_results.csv')


## 6. 最优组合与三次复测

根据单变量实验的真实结果，将有效的 ConvLSTM 改动组合为最优设置，并使用种子 42、43、44 复测三次。复测结果写入 `best_repeat_results.csv`，预测对比图写入 `result_pred.png`。


In [ ]:
# 自动从单变量 ConvLSTM 实验中挑出优于基线的改动，形成可复现的最优组合。
# 若需按实验分析指定组合，可在运行前修改 BEST_CONFIG。
by_id = {r['id']: r for r in all_results}
baseline_mse = by_id['baseline']['test_mse']
BEST_CONFIG = {'id': 'best', 'name': '7 最优组合', 'layers': 1, 'hidden': 32, 'kernel': 3, 'input_frames': 4, 'loss': 'mse'}
for field, exp_id in [('layers', 'exp1'), ('hidden', 'exp2'), ('kernel', 'exp3'), ('input_frames', 'exp4'), ('loss', 'exp6')]:
    if by_id[exp_id]['test_mse'] < baseline_mse:
        BEST_CONFIG[field] = EXPERIMENTS[[e['id'] for e in EXPERIMENTS].index(exp_id)][field]

print('自动选择的最优组合：', BEST_CONFIG)
best_runs, best_outputs = [], None
for repeat_seed in (42, 43, 44):
    result, x, y, pred = run_experiment(BEST_CONFIG, run_seed=repeat_seed)
    best_runs.append(result)
    if repeat_seed == 42:
        best_outputs = (x, y, pred)

mean_result = {k: float(np.mean([r[k] for r in best_runs])) for k in ('test_mse', 'test_mae', 'seconds', 'parameters')}
with open(RESULT_DIR / 'best_repeat_results.csv', 'w', newline='', encoding='utf-8-sig') as f:
    writer = csv.DictWriter(f, fieldnames=['run', 'seed', 'test_mse', 'test_mae', 'seconds', 'parameters'])
    writer.writeheader()
    for i, r in enumerate(best_runs, 1):
        writer.writerow({'run': f'第{i}次', **{k: r[k] for k in writer.fieldnames if k not in ('run',)}})
    writer.writerow({'run': '平均值', 'seed': '', **mean_result})

save_prediction_figure(*best_outputs, RESULT_DIR / 'result_pred.png', BEST_CONFIG['input_frames'])
with open(RESULT_DIR / 'best_config.json', 'w', encoding='utf-8') as f:
    json.dump({'best_config': BEST_CONFIG, 'three_run_average': mean_result}, f, ensure_ascii=False, indent=2)
print('最优组合三次平均：', mean_result)


## 7. 正式结果与讨论

正式运行中，基线测试 MSE 为 0.006012；单变量实验中，5×5 卷积核的 MSE 为 0.004582，64 隐藏通道的 MSE 为 0.004986。展平后的全连接 LSTM 虽有更多参数，但 MSE 为 0.009378，说明保留二维局部空间结构对本任务重要。

最优组合使用 2 层 ConvLSTM、64 隐藏通道、5×5 卷积核和 8 个输入帧。三次复测的平均 MSE 为 0.003967，平均 MAE 为 0.013369。结果说明增加模型容量与感受野能够改善简单运动的下一帧预测，但也会增加训练时间。

本实验只包含单个、规则运动的二值小球；真实视频或雷达回波还会有噪声、目标形态变化、多尺度运动和误差累积，因此这些结果不能直接推广到复杂实际场景。
